# YOLO Fire Detector - Persistent Cloud Training

Questo notebook usa una root persistente unica e mantiene nello stesso punto:

- codice del progetto

- dataset generati e relativi manifest YAML

- checkpoint di training

- modelli esportati e registry degli export


Su Colab monta Google Drive e riparte automaticamente dallo stesso stato se esiste gia' un checkpoint o un dataset compatibile.


In [ ]:
import json

import os

from pathlib import Path

import shutil

import sys

import zipfile



IN_COLAB = 'google.colab' in sys.modules

IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ



if IN_COLAB:

    from google.colab import drive

    drive.mount('/content/drive')

    PERSISTENT_ROOT = Path('/content/drive/MyDrive/yolo-fire-detector')

elif IN_KAGGLE:

    PERSISTENT_ROOT = Path('/kaggle/working/yolo-fire-detector')

else:

    PERSISTENT_ROOT = Path('/content/yolo-fire-detector')



REPO_ROOT = PERSISTENT_ROOT / 'repo'

INPUTS_ROOT = PERSISTENT_ROOT / 'inputs'

RUNTIME_CONFIG_PATH = REPO_ROOT / 'configs' / 'cloud.runtime.yaml'



PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)

INPUTS_ROOT.mkdir(parents=True, exist_ok=True)



print('IN_COLAB =', IN_COLAB)

print('IN_KAGGLE =', IN_KAGGLE)

print('PERSISTENT_ROOT =', PERSISTENT_ROOT)

print('REPO_ROOT =', REPO_ROOT)

print('RUNTIME_CONFIG_PATH =', RUNTIME_CONFIG_PATH)


## Sync progetto nella root persistente



Il notebook legge e scrive sempre nella stessa root persistente. Se trova uno zip aggiornato del progetto, lo estrae nella cartella `repo`; altrimenti riusa quello che c'e' gia'.


In [ ]:
FORCE_PROJECT_REFRESH = False

ZIP_CANDIDATES = [

    INPUTS_ROOT / 'yolo-fire-detector-cloud.zip',

    Path('/content/yolo-fire-detector-cloud.zip'),

    Path('/kaggle/input/yolo-fire-detector-cloud/yolo-fire-detector-cloud.zip'),

]



zip_path = next((candidate for candidate in ZIP_CANDIDATES if candidate.exists()), None)

repo_ready = (REPO_ROOT / 'run_experiment.py').exists()



if zip_path is None and not repo_ready:

    raise FileNotFoundError('Carica yolo-fire-detector-cloud.zip in /content oppure in Drive/inputs prima di proseguire.')



if zip_path is not None and (FORCE_PROJECT_REFRESH or not repo_ready):

    temp_extract_dir = PERSISTENT_ROOT / '_incoming_repo'

    shutil.rmtree(temp_extract_dir, ignore_errors=True)

    shutil.rmtree(REPO_ROOT, ignore_errors=True)

    temp_extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as archive:

        archive.extractall(temp_extract_dir)

    extracted_roots = [child for child in temp_extract_dir.iterdir() if child.is_dir()]

    source_root = extracted_roots[0] if extracted_roots else temp_extract_dir

    shutil.copytree(source_root, REPO_ROOT, dirs_exist_ok=True)

    shutil.rmtree(temp_extract_dir, ignore_errors=True)



print('Project zip =', zip_path)

print('Repo pronta =', REPO_ROOT.exists())

print('run_experiment.py presente =', (REPO_ROOT / 'run_experiment.py').exists())


In [ ]:
import subprocess



os.chdir(REPO_ROOT)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

subprocess.run(['nvidia-smi'], check=False)

print('Dipendenze installate e working directory impostata su', REPO_ROOT)


In [ ]:
import yaml


BASE_CONFIG_NAME = 'cloud.default.yaml'

AVAILABLE_BASE_CONFIGS = [
    'cloud.default.yaml',
    'cloud.quick-screen.yaml',
    'cloud.recall.yaml',
    'cloud.robust.yaml',
    'cloud.small-fire.yaml',
    'cloud.hard-negatives.yaml',
    'cloud.capacity.yaml',
]

DATASET_LABEL = 'synthetic-fire-cloud'

TRAINING_LABEL = 'yolov8n-cloud'

NUM_IMAGES = 2000

IMAGE_SIZE = 640

EPOCHS = 50

BATCH_SIZE = 16

MODEL_SIZE = 'n'

DEVICE = '0'


base_config_path = REPO_ROOT / 'configs' / BASE_CONFIG_NAME

if BASE_CONFIG_NAME not in AVAILABLE_BASE_CONFIGS:
    raise ValueError(f'Preset non supportato: {BASE_CONFIG_NAME}')

if not base_config_path.exists():
    raise FileNotFoundError(f'Config base non trovata: {base_config_path}')

with open(base_config_path, 'r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)


config['project']['persistent_root'] = str(PERSISTENT_ROOT)

config['dataset']['label'] = DATASET_LABEL

config['dataset']['num_images'] = NUM_IMAGES

config['dataset']['image_size'] = IMAGE_SIZE

config['dataset']['force_regenerate'] = False

config['training']['label'] = TRAINING_LABEL

config['training']['model_size'] = MODEL_SIZE

config['training']['epochs'] = EPOCHS

config['training']['batch_size'] = BATCH_SIZE

config['training']['image_size'] = IMAGE_SIZE

config['training']['device'] = DEVICE

config['training']['resume'] = 'auto'


RUNTIME_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(RUNTIME_CONFIG_PATH, 'w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False, allow_unicode=False)


print('BASE_CONFIG_NAME =', BASE_CONFIG_NAME)
print(RUNTIME_CONFIG_PATH.read_text(encoding='utf-8'))

In [ ]:
command = [sys.executable, 'run_experiment.py', '--config', str(RUNTIME_CONFIG_PATH)]

print('Executing:', ' '.join(command))

subprocess.run(command, check=True, cwd=REPO_ROOT)


## Output persistenti


La pipeline salva sempre nella stessa root persistente:

- `datasets/` con cartelle versionate per fingerprint e `dataset_manifest.yaml`

- `runs/` con config risolta, summary YAML, metriche e diagnostica YOLO

- `exports/` con export finale etichettato e `latest.yaml`


Se il dataset richiesto esiste gia' con gli stessi parametri viene riusato; se la run ha `weights/last.pt`, il resume avviene automaticamente.


In [ ]:
latest_registry = PERSISTENT_ROOT / 'exports' / 'latest.yaml'

if latest_registry.exists():

    print(latest_registry.read_text(encoding='utf-8'))

else:

    print('Nessun export registrato ancora.')


print('\nDatasets disponibili:')

for path in sorted((PERSISTENT_ROOT / 'datasets').glob('*')):

    print('-', path)


print('\nRun disponibili:')

for path in sorted((PERSISTENT_ROOT / 'runs').glob('*')):

    print('-', path)